In [ ]:
# Uncomment and run this cell ONCE, then restart runtime

!pip install -q -U transformers accelerate peft bitsandbytes datasets
!pip install -q evaluate bert-score rouge-score sacrebleu
!pip install -q pandas matplotlib seaborn tqdm


In [ ]:
import os
import gc
import json
import torch
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm
from typing import Dict, List, Optional

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training
)
from datasets import Dataset
import evaluate

# Configuration and Setup

In [ ]:
class Config:
    """Central configuration for the training pipeline"""

    # Google Drive paths
    DRIVE_ROOT = "/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/"
    DATA_PATH = f"{DRIVE_ROOT}/data/processed/buddhist_corpus_combined_20251101_105228.txt"
    CHECKPOINT_DIR = f"{DRIVE_ROOT}/model/checkpoints/sinllama_pretrain"
    RESULTS_DIR = f"{DRIVE_ROOT}/model/results/pretrain_results"

    # Model configuration
    BASE_MODEL = "meta-llama/Meta-Llama-3-8B"
    SINLLAMA_MODEL = "polyglots/SinLlama_v01"

    # Training hyperparameters
    BATCH_SIZE = 4  # Adjust based on your GPU memory
    GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 4 * 4 = 16
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 3
    MAX_SEQ_LENGTH = 512
    WARMUP_STEPS = 100
    SAVE_STEPS = 500
    EVAL_STEPS = 500
    LOGGING_STEPS = 100

    # LoRA configuration
    LORA_R = 16  # Rank of LoRA matrices
    LORA_ALPHA = 32  # Scaling parameter
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

    # 4-bit quantization config
    USE_4BIT = True
    BNB_4BIT_COMPUTE_DTYPE = torch.float16
    BNB_4BIT_QUANT_TYPE = "nf4"
    USE_NESTED_QUANT = True

    # Evaluation config
    EVAL_BATCH_SIZE = 8
    NUM_EVAL_SAMPLES = 100  # Number of samples for evaluation

# Initialize config
cfg = Config()

# Create directories
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)

print("✅ Configuration loaded successfully!")
print(f"📁 Checkpoints will be saved to: {cfg.CHECKPOINT_DIR}")
print(f"📊 Results will be saved to: {cfg.RESULTS_DIR}")

# Mount Google Drive and Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def load_text_dataset(file_path: str, sample_size: Optional[int] = None) -> List[str]:
    """
    Load text dataset from file.

    Args:
        file_path: Path to text file
        sample_size: Optional limit on number of lines to load

    Returns:
        List of text strings
    """
    print(f"📖 Loading dataset from: {file_path}")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ File not found: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        texts = f.readlines()

    # Remove empty lines and strip whitespace
    texts = [line.strip() for line in texts if line.strip()]

    if sample_size:
        texts = texts[:sample_size]

    print(f"✅ Loaded {len(texts):,} text samples")
    print(f"📏 Average length: {np.mean([len(t) for t in texts]):.1f} characters")

    return texts

def preprocess_dataset(
    texts: List[str],
    tokenizer,
    max_length: int = 512,
    train_split: float = 0.9
) -> Dict[str, Dataset]:
    """
    Preprocess and tokenize text dataset.

    Args:
        texts: List of text strings
        tokenizer: Hugging Face tokenizer
        max_length: Maximum sequence length
        train_split: Fraction of data for training

    Returns:
        Dictionary with 'train' and 'eval' datasets
    """
    print(f"🔄 Preprocessing dataset...")

    # Create DataFrame
    df = pd.DataFrame({'text': texts})

    # Split into train and eval
    train_size = int(len(df) * train_split)
    train_df = df[:train_size]
    eval_df = df[train_size:]

    print(f"📊 Train samples: {len(train_df):,}")
    print(f"📊 Eval samples: {len(eval_df):,}")

    # Convert to Hugging Face datasets
    train_dataset = Dataset.from_pandas(train_df)
    eval_dataset = Dataset.from_pandas(eval_df)

    # Tokenization function
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=max_length,
            padding='max_length',
            return_tensors='pt'
        )

    # Tokenize datasets
    print("🔤 Tokenizing datasets...")
    train_dataset = train_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=['text'],
        desc="Tokenizing train data"
    )

    eval_dataset = eval_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=['text'],
        desc="Tokenizing eval data"
    )

    print("✅ Preprocessing complete!")

    return {
        'train': train_dataset,
        'eval': eval_dataset
    }

# Load your dataset
texts = load_text_dataset(cfg.DATA_PATH)

# Display sample
print("\n📝 Sample texts:")
for i, text in enumerate(texts[:3]):
    print(f"\n{i+1}. {text[:200]}...")

# Load SinLlama Model with 4-bit Quantization

In [ ]:
def load_sinllama_model(model_id: str, cfg: Config):
    """
    Load SinLlama model with 4-bit quantization and prepare for training.

    Args:
        model_id: Hugging Face model ID
        cfg: Configuration object

    Returns:
        model, tokenizer, peft_config
    """
    print(f"🦙 Loading SinLlama model: {model_id}")

    # Configure 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=cfg.USE_4BIT,
        bnb_4bit_compute_dtype=cfg.BNB_4BIT_COMPUTE_DTYPE,
        bnb_4bit_quant_type=cfg.BNB_4BIT_QUANT_TYPE,
        bnb_4bit_use_double_quant=cfg.USE_NESTED_QUANT
    )

    # Load tokenizer
    print("📚 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Load base model with quantization
    print("🔄 Loading base model with 4-bit quantization...")
    model = AutoModelForCausalLM.from_pretrained(
        cfg.BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=cfg.BNB_4BIT_COMPUTE_DTYPE,
        low_cpu_mem_usage=True
    )

    # Resize embeddings to match SinLlama's extended vocabulary
    print("🔧 Resizing embeddings for extended vocabulary...")
    model.resize_token_embeddings(len(tokenizer))

    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model)

    # Configure LoRA
    print("⚡ Configuring LoRA adapters...")
    peft_config = LoraConfig(
        r=cfg.LORA_R,
        lora_alpha=cfg.LORA_ALPHA,
        lora_dropout=cfg.LORA_DROPOUT,
        target_modules=cfg.LORA_TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply LoRA
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # Load SinLlama adapters
    print("🔄 Loading SinLlama PEFT adapters...")
    try:
        model = PeftModel.from_pretrained(
            model,
            model_id,
            is_trainable=True
        )
        print("✅ SinLlama adapters loaded successfully!")
    except Exception as e:
        print(f"⚠️ Warning: Could not load SinLlama adapters: {e}")
        print("Proceeding with base LoRA configuration...")

    # Display memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"\n💾 GPU Memory:")
        print(f"   Allocated: {allocated:.2f} GB")
        print(f"   Reserved: {reserved:.2f} GB")

    return model, tokenizer, peft_config

# Load model
model, tokenizer, peft_config = load_sinllama_model(cfg.SINLLAMA_MODEL, cfg)

# Preprocess dataset
datasets = preprocess_dataset(
    texts,
    tokenizer,
    max_length=cfg.MAX_SEQ_LENGTH
)

# Pre-Training Evaluation (Baseline)

In [ ]:
class SinLlamaEvaluator:
    """Evaluator for measuring model performance"""

    def __init__(self, model, tokenizer, device='cuda'):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

        # Load metrics
        self.perplexity_metric = evaluate.load('perplexity')

    def calculate_perplexity(self, eval_dataset, max_samples: int = 100):
        """Calculate perplexity on evaluation dataset"""
        print(f"📊 Calculating perplexity on {max_samples} samples...")

        # Sample dataset
        if len(eval_dataset) > max_samples:
            indices = np.random.choice(len(eval_dataset), max_samples, replace=False)
            eval_subset = eval_dataset.select(indices)
        else:
            eval_subset = eval_dataset

        # Decode texts
        texts = []
        for item in tqdm(eval_subset, desc="Decoding texts"):
            text = self.tokenizer.decode(item['input_ids'], skip_special_tokens=True)
            texts.append(text)

        # Calculate perplexity
        self.model.eval()
        with torch.no_grad():
            results = self.perplexity_metric.compute(
                predictions=texts,
                model_id=cfg.BASE_MODEL
            )

        return results['mean_perplexity']

    def generate_samples(self, prompts: List[str], max_length: int = 100):
        """Generate text samples for qualitative evaluation"""
        print(f"🎨 Generating {len(prompts)} samples...")

        self.model.eval()
        generations = []

        for prompt in tqdm(prompts, desc="Generating"):
            inputs = self.tokenizer(prompt, return_tensors='pt').to(self.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_length=max_length,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            generations.append({
                'prompt': prompt,
                'generation': generated_text
            })

        return generations

# Initialize evaluator
evaluator = SinLlamaEvaluator(model, tokenizer)

# Baseline evaluation
print("\n" + "="*60)
print("📊 BASELINE EVALUATION (Before Training)")
print("="*60)

baseline_perplexity = evaluator.calculate_perplexity(
    datasets['eval'],
    max_samples=cfg.NUM_EVAL_SAMPLES
)
print(f"📈 Baseline Perplexity: {baseline_perplexity:.2f}")

# Generate baseline samples
test_prompts = [
    "සිද්ධාර්ථ ගෞතම බුදුන් වහන්සේ",  # Sinhala
    "The Four Noble Truths are",  # English
    "ධර්මය කියන්නේ"  # Sinhala
]

baseline_generations = evaluator.generate_samples(test_prompts)

print("\n📝 Baseline Generations:")
for i, gen in enumerate(baseline_generations):
    print(f"\n{i+1}. Prompt: {gen['prompt']}")
    print(f"   Generation: {gen['generation'][:200]}...")

# Continual Pre-training

In [ ]:
# Configure training arguments
training_args = TrainingArguments(
    output_dir=cfg.CHECKPOINT_DIR,
    num_train_epochs=cfg.NUM_EPOCHS,
    per_device_train_batch_size=cfg.BATCH_SIZE,
    per_device_eval_batch_size=cfg.EVAL_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    warmup_steps=cfg.WARMUP_STEPS,
    logging_steps=cfg.LOGGING_STEPS,
    save_steps=cfg.SAVE_STEPS,
    eval_steps=cfg.EVAL_STEPS,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    save_total_limit=3,
    report_to="none",
    push_to_hub=False,
    remove_unused_columns=False
)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM, not masked LM
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['eval'],
    data_collator=data_collator
)

# Start training
print("\n" + "="*60)
print("🚀 STARTING CONTINUAL PRE-TRAINING")
print("="*60)
print(f"📊 Training samples: {len(datasets['train']):,}")
print(f"📊 Evaluation samples: {len(datasets['eval']):,}")
print(f"⚡ Effective batch size: {cfg.BATCH_SIZE * cfg.GRADIENT_ACCUMULATION_STEPS}")
print(f"🔄 Total steps: {len(datasets['train']) // (cfg.BATCH_SIZE * cfg.GRADIENT_ACCUMULATION_STEPS) * cfg.NUM_EPOCHS}")

# Train
train_result = trainer.train()

# Save final model
final_model_path = f"{cfg.CHECKPOINT_DIR}/final_model"
trainer.save_model(final_model_path)
print(f"\n✅ Final model saved to: {final_model_path}")

# Save training metrics
metrics_path = f"{cfg.RESULTS_DIR}/training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(train_result.metrics, f, indent=2)
print(f"📊 Training metrics saved to: {metrics_path}")

# Post-Training Evaluation

In [ ]:
print("\n" + "="*60)
print("📊 POST-TRAINING EVALUATION")
print("="*60)

# Reload best model
best_checkpoint = trainer.state.best_model_checkpoint
print(f"🔄 Loading best checkpoint: {best_checkpoint}")

# Calculate post-training perplexity
post_perplexity = evaluator.calculate_perplexity(
    datasets['eval'],
    max_samples=cfg.NUM_EVAL_SAMPLES
)
print(f"📈 Post-training Perplexity: {post_perplexity:.2f}")

# Generate post-training samples
post_generations = evaluator.generate_samples(test_prompts)

print("\n📝 Post-training Generations:")
for i, gen in enumerate(post_generations):
    print(f"\n{i+1}. Prompt: {gen['prompt']}")
    print(f"   Generation: {gen['generation'][:200]}...")

# Results Comparison and Visualization

In [ ]:
# Compile results
results = {
    'timestamp': datetime.now().isoformat(),
    'config': {
        'model': cfg.SINLLAMA_MODEL,
        'epochs': cfg.NUM_EPOCHS,
        'learning_rate': cfg.LEARNING_RATE,
        'batch_size': cfg.BATCH_SIZE * cfg.GRADIENT_ACCUMULATION_STEPS,
        'lora_r': cfg.LORA_R,
        'lora_alpha': cfg.LORA_ALPHA
    },
    'baseline': {
        'perplexity': baseline_perplexity,
        'generations': baseline_generations
    },
    'post_training': {
        'perplexity': post_perplexity,
        'generations': post_generations
    },
    'improvement': {
        'perplexity_reduction': baseline_perplexity - post_perplexity,
        'perplexity_reduction_pct': ((baseline_perplexity - post_perplexity) / baseline_perplexity) * 100
    }
}

# Save complete results
results_path = f"{cfg.RESULTS_DIR}/evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n✅ Complete results saved to: {results_path}")

# Display summary
print("\n" + "="*60)
print("📊 TRAINING SUMMARY")
print("="*60)
print(f"\n📉 Perplexity Improvement:")
print(f"   Before: {baseline_perplexity:.2f}")
print(f"   After:  {post_perplexity:.2f}")
print(f"   Change: {results['improvement']['perplexity_reduction']:.2f} ({results['improvement']['perplexity_reduction_pct']:.2f}% reduction)")

print("\n📁 Saved Artifacts:")
print(f"   ✅ Final model: {final_model_path}")
print(f"   ✅ Training metrics: {metrics_path}")
print(f"   ✅ Evaluation results: {results_path}")
print(f"   ✅ Checkpoints: {cfg.CHECKPOINT_DIR}")

# Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Create comparison plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

categories = ['Baseline', 'After Training']
perplexities = [baseline_perplexity, post_perplexity]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(categories, perplexities, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Perplexity (Lower is Better)', fontsize=12, fontweight='bold')
ax.set_title('Model Performance: Before vs After Continual Pre-training',
             fontsize=14, fontweight='bold', pad=20)
ax.set_ylim(0, max(perplexities) * 1.2)

plt.tight_layout()
plot_path = f"{cfg.RESULTS_DIR}/perplexity_comparison.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"\n📊 Visualization saved to: {plot_path}")
plt.show()